# Case Study 05: Deciding what an independent creator should copy from the small channels that trended twice

**Analysis layer.** Every figure in the prose below is **interpolated at build time** from
`2-analysis/out/*.json`, which the six question scripts wrote, and **re-asserted at execution** by
the integrity gate in the final appendix — so a script re-run without a notebook rebuild fails loudly
instead of drifting. Two code cells go further and recompute from the shaped parquet, to prove the
JSON is not stale. Numbers and their uncertainty live here; the business recommendation does not —
that is the storyteller's layer, and its absence from this notebook is deliberate.

*Basis:* 6,250 trending videos, each measured at its **first** trending snapshot,
US board, 2017-11-14 to 2018-06-14.

## Problem Statement

An independent creator runs a channel alone. They choose what gets made and when it ships, and
nobody above them approves either. They believe what most creators believe — that trending is the
algorithm's business and luck's, and that very little they control moves it. They have said so
plainly, and they have said what would change their mind: evidence from channels their own size,
not from the Music and Entertainment giants whose numbers swamp every aggregate.

The number they are judged on is engagement rate: likes plus comments, per view. It is what a
prospective sponsor reads when deciding whether the channel is worth backing, and the creator
believes it is what the platform reads too.

They came to the BI team with a specific version of that question. Some small channels reach the
US trending board once and are never seen there again. Others come back. If something separates
those two groups — something visible on the videos themselves — then it is a thing a creator could
copy, and the belief that none of this is controllable would be wrong in a useful way.

This case study tests that. It works from 205 days of the US trending board and nothing else: no
subscriber counts, no watch time, no click-through, no view of the uploads that never trended. What
those missing columns cost the answer is stated where it bites, not collected in a footnote.

## Initial Analysis

Before the question, the data. What one row is, what is wrong with it, what had to be built, and
what the two headline metrics actually mean once they are defined precisely.

### Setup

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 190)
pd.set_option("display.max_columns", 50)

ROOT = Path.cwd().parents[0] if Path.cwd().name == "scripts" else Path.cwd()
if not (ROOT / "Analysis").exists():
    ROOT = ROOT.parent
OUT, DATA = ROOT / "Analysis" / "out", ROOT / "Analysis" / "data"

J = {p.stem: json.loads(p.read_text()) for p in OUT.glob("*.json")}
ruled = pd.read_parquet(DATA / "videos_ruled.parquet")
smaller = ruled[ruled["scale_group"] == "smaller"]

print("ruled population      ", ruled.shape)
print("smaller-channel subset", smaller.shape)
ruled.info(verbose=False)

ruled population       (6250, 45)
smaller-channel subset (3489, 45)
<class 'pandas.DataFrame'>
RangeIndex: 6250 entries, 0 to 6249
Columns: 45 entries, video_id to is_channel_first
dtypes: bool(12), datetime64[us, America/New_York](1), datetime64[us](5), float64(4), int32(1), int64(8), str(14)
memory usage: 10.5 MB


### Explore + Profile

**The grain, narrated.** One row of the raw file is *one video on one trending day*, and its
`views`, `likes` and `comment_count` are cumulative **from upload**, not from that day. A video
that held the board for thirty days contributes thirty rows, each restating its running totals. So
nothing sums across rows, and every per-video statistic must first pick exactly one row per video.

That single decision is the largest methodological fork in the study, and the plan resolved it
before any compute: **the first trending snapshot**, so every video is compared at a comparable
age rather than at whatever age its own run happened to end.

In [2]:
d = J["cleaning-log"]
print("RAW  rows", d["verification"]["rows_in"], "-> CLEAN rows", d["verification"]["rows_out"])
print("videos", d["verification"]["videos"], "| trending days", d["verification"]["trending_days"])
print()
print("One video, its first three board days — the cumulative restatement:")
v = ruled.iloc[0]["video_id"]
full = pd.read_parquet(DATA / "clean_video_day.parquet")
g = full[full["video_id"] == v].sort_values("trending_day").head(3)
print(g[["trending_day", "views", "likes", "comment_count"]].to_string(index=False))
print()
print("Sentinel guard — pandas' default reader would swallow these:")
print(f"  tags == '[none]'      {int((ruled['tag_count'] == 0).sum())} ruled videos (a category, not a null)")
print(f"  description == ''     {int((~ruled['has_description']).sum())} ruled videos")

RAW  rows 40949 -> CLEAN rows 40899
videos 6351 | trending days 205

One video, its first three board days — the cumulative restatement:
trending_day  views  likes  comment_count
  2018-06-06 475965   6531            271
  2018-06-07 605506   7848            354
  2018-06-08 705986   8930            371

Sentinel guard — pandas' default reader would swallow these:
  tags == '[none]'      251 ruled videos (a category, not a null)
  description == ''     89 ruled videos


### Clean

Triage, not perfection: fix what is fixable without a stakeholder, document the rest with its
magnitude, impute nothing. Every decision below is logged in `out/cleaning-log.json`.

**Fixed.** Sentinel-safe load (recovers 570
`description` values the default reader coerces to null) · `trending_date` parsed from `YY.DD.MM` ·
**`publish_time` converted UTC → America/New_York**, which moves the publish hour for every row and
the publish *date* for 12.51%
of them · category code joined to name · **50
duplicate rows removed** (0.122% of the file),
keeping maximum views.

**Documented and left**, because fixing any of them would need context this file does not carry:
8 missing calendar days (3.76% of the span) ·
mid-run metadata churn (up to 9.9% of videos
edited something) · trending runs censored at both window edges ·
4 videos flagged removed.

The churn deserves a note, because it is the one that could have gone wrong quietly: 191 videos
were retagged and 109 retitled while trending. Taking a per-video attribute as a *mode* across its
rows would silently mix pre- and post-edit metadata. Fixing one row per video removes the problem
rather than managing it.

In [3]:
cl = J["cleaning-log"]
rows = [("SOLVABLE — " + k, v.get("pct_of_rows", v.get("rows_whose_hour_changes_pct", "—")))
        for k, v in cl["solvable"].items()]
rows += [("LEFT — " + k, v.get("pct_of_span", v.get("max_pct_affected", v.get("pct_either", v.get("pct", "—")))))
         for k, v in cl["unsolvable"].items()]
print(pd.DataFrame(rows, columns=["decision", "magnitude_%"]).to_string(index=False))

                             decision magnitude_%
     SOLVABLE — S1_sentinel_safe_read        1.39
   SOLVABLE — S2_trending_date_parsed           —
 SOLVABLE — S3_publish_time_utc_to_et       100.0
        SOLVABLE — S4_category_lookup           —
        SOLVABLE — S5_dedup_video_day       0.122
      LEFT — U1_missing_calendar_days        3.76
             LEFT — U2_metadata_churn         9.9
              LEFT — U3_censored_runs         6.3
           LEFT — U4_error_or_removed       0.063
LEFT — U5_like_ratio_zero_denominator       0.016


### Shape

Only fields a PRD question or north-star consumes. Two cut-points carry real weight and both are
justified rather than chosen for convenience:

- **Scale split at a channel median of 354,264 views** — the human's own threshold, confirmed at the
  gate and left untouched. It puts **3,489 of 6,250 videos (55.8%)**
  and **1,385 of 2,099 channels (66.0%)**
  on the smaller side. Two thirds of the channels on this board — which is why the earlier label
  "bottom half" was retired: a sponsor hearing "bottom half" pictures a minority, and this is not one.
- **Publish blocks 14:00–18:59 vs 19:00–23:59 ET** — widened from a narrower pair because only at
  this width do per-category cells clear n≥30 on both sides. The bar was not lowered to reach more
  categories; the window was widened and the coverage cost reported.

**The back-catalogue exclusion is a basis decision, not a data fix.** Videos taking more than 30
days to reach the board leave the engagement population — 101 videos
(1.59%). It is what makes "fixed maturity" true rather than asserted: maximum age
at measurement falls from 4,215 days to
28.

In [4]:
sl = J["shape-log"]
print("DERIVED FIELDS (name -> which question consumes it)")
for f in sl["derived_fields"]:
    print(f"  {f['name']:34s} {f['feeds']}")
print()
print("CUT-POINT REASONS")
for k, v in sl["bucket_cut_reasons"].items():
    print(f"  {k}: {v}")
print()
print("POST-SHAPE SANITY (re-run after shaping — augmentation surfaces fresh bad rows)")
for k, v in sl["post_shape_checks"].items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
assert all(sl["post_shape_checks"].values())

DERIVED FIELDS (name -> which question consumes it)
  run_days                           Q1 (supporting KPI), Q4
  censored_left/right/any            Q1, Q4 — run days is censored
  days_to_trend                      Q1, the back-catalogue exclusion
  back_catalogue                     the exclusion itself
  rate_eligible                      Q4 (its own line), every rate
  eng_rate                           north-star 1 — Q1-Q6
  like_ratio                         supporting KPI — Q1, Q4
  ch_med_views                       the scale split
  scale_group                        Q3, Q5, Q6
  publish_hour_et / publish_dow_et   Q4, Q5, Q6
  publish_block                      Q5 (the priced move), Q6
  publish_block_narrow               Q5 method note
  tag_count                          Q4, Q6
  has_tags / has_description         Q4, Q6
  trending_month / trending_week     Q1 stability
  channel_n_videos                   Q6
  channel_group                      Q6
  is_channel_first       

### Metric definitions and their benchmarks

**Engagement rate** = `(likes + comment_count) / views * 100`, taken as the **median over videos**,
never pooled. Pooling — total reactions over total views — would hand the answer to the largest
channels, which are exactly the ones this reader discounts. Videos with comments or ratings
disabled have a structurally truncated numerator and carry no rate at all
(109 of 6,250 ruled videos); they are
reported as their own line rather than counted as zero.

**Reach** = median views per video at the same snapshot. It is the price: what engagement is traded
against.

Every benchmark here is internal. No industry figure is quoted, because none is comparable to a
board-only population measured at first appearance.

In [5]:
b = J["basis"]
print(f"ALL RULED        {b['all_ruled']['eng_median_pct']}% engagement on {b['all_ruled']['views_median']:,} views"
      f"   (IQR {b['all_ruled']['eng_q1_pct']}-{b['all_ruled']['eng_q3_pct']})")
print(f"SMALLER CHANNELS {b['smaller_channels']['eng_median_pct']}% on {b['smaller_channels']['views_median']:,} views")
print()
print("Recomputed live from the parquet, to prove the JSON is not stale:")
print(f"  all ruled   {ruled['eng_rate'].median():.4f}%  on {ruled['views'].median():,.0f} views")
print(f"  smaller     {smaller['eng_rate'].median():.4f}%  on {smaller['views'].median():,.0f} views")
print()
print(f"CONVERSION — 1 percentage point of engagement = "
      f"{b['conversion']['reactions_per_point_per_upload']:,} likes+comments per upload at the")
print("smaller-channel baseline. Every percentage point below converts at that rate.")

ALL RULED        3.7331% engagement on 279,895 views   (IQR 1.8912-6.3218)
SMALLER CHANNELS 3.5349% on 119,028 views

Recomputed live from the parquet, to prove the JSON is not stale:
  all ruled   3.7331%  on 279,896 views
  smaller     3.5349%  on 119,028 views

CONVERSION — 1 percentage point of engagement = 1,190 likes+comments per upload at the
smaller-channel baseline. Every percentage point below converts at that rate.


---

## The Central Question

> **One-hit or repeat — what separates a small channel that trends twice?**

Chosen by the decision-owner at the specification gate, from five candidates each presented with
its likely headline and its price. The runner-up — what an afternoon publishing tilt costs inside a
category — keeps its figures and loses the headline, because the human declined that move once its
price was finally stated correctly.

The ladder below earns the answer. Each question carries a verb type and a role in the argument:

| # | Verb | Question | Role |
|---|---|---|---|
| Q1 | Size | What does a typical trending video score, and how wide is the spread? | context |
| Q2 | Rank | Which categories lead and lag, and how far is that from the reach ranking? | BUILD |
| Q3 | Compare | Does that ranking hold among smaller channels? | BUILD/TEST |
| Q4 | Explain | Of the dials a creator owns, which is associated with engagement — and how much spread is left over? | **BREAK** |
| Q5 | Compare | Which calendar moves are associated with higher engagement, and what does each cost in views? | TURN |
| Q6 | Recommend | What separated the channels that trended more than once? | **RESOLVE** |

**Depth is bought, not sprayed.** Q1–Q5 run three rungs: a primary estimate with an interval, an
assumption check *before* the method, and one named alternative explanation killed with a number.
**Q6 alone carries a fourth rung**, reallocated to it by the human at the gate, and it buys exactly
two things: the descriptor set and the sensitivity behind a pre-declared fallback.

One structural note that governs every interval in Q1–Q5. Videos are not independent draws: 3,489
smaller-channel videos come from 1,385 channels, and the 450 repeat channels carry 2,554 of them.
A bootstrap that resamples videos would report intervals that are too narrow. Every interval below
resamples **channels**. Q6's design is exempt — it uses exactly one video per channel — and it
proves that rather than asserting it.

## Q1 (Size) — What a typical trending video scores, and how wide the spread is

**Question.** What does a typical trending video score on engagement and reach at a fixed maturity, and how wide is the spread?

**Verb type:** Size · **Arc role:** context

**Approach.** Median, not mean, and a bootstrap interval rather than a normal-theory one. The rejected alternative is the mean with a t-interval: engagement is right-skewed and views are extreme (the top 1% of videos hold roughly a fifth of all views), so a mean would report the whales and an interval built on normality would be wrong about its own width.

**Assumption check — run before the method.** Two checks. Shape first — eng_rate is right-skewed (all-ruled skew 1.735, mean 1.263× the median) and views is extreme (skew 10.927, mean 2.75× the median, top 1% of videos holding 19.45% of all views) — a mean would report the whales, so the PRD's pre-declared median of per-video values stands, and pooling sum(reactions)/sum(views) is barred for the same reason. Then independence, which decides the *interval* rather than the estimator: the intra-channel correlation is 0.7779, a design effect of 2.2. That is large enough that a video-level bootstrap would be misleading, so the interval is a channel-clustered bootstrap — and the correction is validated against a permutation null rather than assumed.

In [6]:
d = J["q1_baseline"]
e, a = d["rung1_estimates"]["smaller"]["eng_rate_pct"], d["rung1_estimates"]["all_ruled"]["eng_rate_pct"]
print("ENGAGEMENT RATE, median of per-video rates")
print(f"  smaller channels  {e['median']:.4f}%   clustered CI {e['ci95_clustered']}   n={e['n']:,}")
print(f"  all ruled         {a['median']:.4f}%   clustered CI {a['ci95_clustered']}   n={a['n']:,}")
print()
print("WHY THE CLUSTERED INTERVAL IS THE ONE THAT MAY BE QUOTED")
v = d["rung2_interval_validation"]
print(f"  observed clustered width {v['observed_clustered_width_pp']}pp")
print(f"  naive video-bootstrap    {v['naive_width_pp']}pp")
print(f"  clustered width under a PERMUTED null {v['permuted_null_clustered_width_pp_mean']}pp")
print(f"  -> {v['verdict']}")
print()
print("THE SPREAD, in the reader's own units")
print(f"  p10 {e['p10']}%  p25 {e['p25']}%  median {e['median']:.2f}%  p75 {e['p75']}%  p90 {e['p90']}%")
r = d["rollup1_conversion_factor"]["smaller"]
print(f"  1pp = {r['reactions_per_point_per_upload']:,} reactions/upload -> the p10-to-p90 span is worth")
print(f"       about {(e['p90'] - e['p10']) * r['reactions_per_point_per_upload']:,.0f} reactions on a median upload")

ENGAGEMENT RATE, median of per-video rates
  smaller channels  3.5349%   clustered CI [3.2108, 3.9126]   n=3,425
  all ruled         3.7331%   clustered CI [3.458, 4.0357]   n=6,141

WHY THE CLUSTERED INTERVAL IS THE ONE THAT MAY BE QUOTED
  observed clustered width 0.5948pp
  naive video-bootstrap    0.1958pp
  clustered width under a PERMUTED null 0.1935pp
  -> Under the permutation null the clustered width collapses to 0.1935pp — essentially the naive width (0.1958pp). The observed clustered width of 0.5948pp is therefore real within-channel correlation, not cluster-size heterogeneity. The naive video bootstrap understates the interval by roughly 3.0x and the clustered CI is the one that may be quoted.

THE SPREAD, in the reader's own units
  p10 0.878%  p25 1.716%  median 3.53%  p75 6.0232%  p90 9.1194%
  1pp = 1,190.3 reactions/upload -> the p10-to-p90 span is worth
       about 9,810 reactions on a median upload


**Result with uncertainty.** Smaller channels sit at **3.5349%** (95% CI [3.2108, 3.9126]) on median **119,028 views**. The clustered interval is 2.55× wider than the naive one, and the permutation null shows that gap is real within-channel correlation rather than an artifact of uneven channel sizes.

**Read.** The median is a location, not a norm. The middle 80% of smaller-channel videos runs from 0.878% to 9.1194% — better than a tenfold range, worth roughly 9,810 reactions on a median upload. Any later claim of a one- or two-point effect is a small shift inside a very wide distribution, and should be read that way.

**Rule-out.** **Killed: the spread is just category mix.** If comparing like with like collapsed the range, the baseline would be an artifact of mixing Music with Sports. It does not. Inside a single category the spread barely narrows: median within-category IQR is 3.3855pp against an overall 4.3072pp for smaller channels (78.6% of it), and category accounts for only ε²=0.211 of the rank variance in eng_rate. The spread is a property of the board, not of the mixing.

**Answer, hypothesis, confidence.** **Answer:** the smaller-channel baseline is 3.5349% engagement on 119,028 views, and the spread around it is wide enough that the median describes almost nobody. **Hypothesis (associative, to test):** the surviving spread is associated with *channel identity* rather than with the video — an ICC of 0.7779 says most variation sits between channels, not within them. The test that would confirm it is the within-channel IQR share, built the same way as the category rule-out. **Confidence: high** — every point estimate reproduces the basis exactly, and the interval method was validated rather than asserted.

**One further result belongs here, because it bounds how every engagement figure in this study may
be quoted to a sponsor.** The same videos read differently depending on when they are measured.

In [7]:
s = J["q1_baseline"]["rollup4_sponsor_basis_limit"]["all_ruled"]
print(f"n = {s['n']:,} videos rate-eligible on BOTH bases")
print(f"  engagement, first snapshot  {s['eng_median_first_pct']}%")
print(f"  engagement, last  snapshot  {s['eng_median_last_pct']}%   ({s['eng_median_signed_change_pct']}%)")
print(f"  views,      first -> last   {s['views_median_first']:,.0f} -> {s['views_median_last']:,.0f} "
      f"({s['views_median_signed_change_pct']:+}%)")
print(f"  the two bases rank videos   rho = {s['spearman_rho_first_vs_last_eng']}")
print()
print("The LEVEL depends on when you measure. The ORDERING barely does. That is what makes the")
print("comparisons in this study usable at all -- and it means no figure here is the number a")
print("creator sees on their own lifetime dashboard, which sits below both and cannot be computed")
print("from board data.")

n = 6,132 videos rate-eligible on BOTH bases
  engagement, first snapshot  3.7367%
  engagement, last  snapshot  2.8307%   (-24.25%)
  views,      first -> last   281,650 -> 534,415 (+89.74%)
  the two bases rank videos   rho = 0.9249

The LEVEL depends on when you measure. The ORDERING barely does. That is what makes the
comparisons in this study usable at all -- and it means no figure here is the number a
creator sees on their own lifetime dashboard, which sits below both and cannot be computed
from board data.


## Q2 (Rank) — Where a category sits, and how firmly

**Question.** Which categories lead and lag on engagement, and how far is that from the reach ranking?

**Verb type:** Rank · **Arc role:** BUILD

**Approach.** Kruskal–Wallis with eta-squared for the omnibus, then per-category medians with channel-clustered bootstrap intervals **and bootstrap rank intervals**. Rank intervals matter: a ranking printed as fourteen tidy positions implies a precision the data does not have. The rejected alternative is one-way ANOVA, rejected *before* use — see the check below. This question is a locator, not a league table: the reader cannot change category, so its job is to tell them which row is theirs and what baseline that sets.

**Assumption check — run before the method.** ANOVA was tested and failed: skew 1.7347, normality p=0, between-category variance ratio 8.2031× with Levene p≈2e-124. Method switched to Kruskal–Wallis on ranks. The omnibus then returned η²=0.2481 (CI [0.2096, 0.2987]) — a large effect whose whole interval sits above the 0.14 band, so a real ordering is licensed. What it does **not** licence is any named pair: an omnibus says the medians are not all equal, not that Music beats Education. So every pair carries its own interval and the 91 pairwise gaps are Benjamini–Hochberg corrected.

In [8]:
d = J["q2_category_rank"]
t = pd.DataFrame(d["rollup_1_per_category_table"])
show = t[t["rankable"]].sort_values("eng_rank")[
    ["category", "n_videos", "n_channels", "eng_median_pct", "eng_ci95", "eng_rank", "eng_rank_ci95",
     "reach_rank", "in_covered_six"]]
print(show.to_string(index=False))
print()
fixed = [r["category"] for _, r in t.iterrows() if r["rankable"] and r["eng_rank_ci95"][0] == r["eng_rank_ci95"][1]]
wide = [r["category"] for _, r in t.iterrows() if r["rankable"] and (r["eng_rank_ci95"][1] - r["eng_rank_ci95"][0]) >= 4]
print(f"categories holding a FIXED bootstrap rank: {len(fixed)} -> {fixed}")
print(f"categories whose rank interval is >=4 places wide: {len(wide)} of {int(t['rankable'].sum())}")
print()
s = d["rollup_3_separation"]
print(f"pairwise gaps {s['n_pairwise_comparisons']} | CI excludes zero {s['n_ci_excludes_zero']} "
      f"| survive BH q=0.05 {s['n_bh_significant_q05']}")

            category  n_videos  n_channels  eng_median_pct         eng_ci95  eng_rank eng_rank_ci95  reach_rank  in_covered_six
               Music       790         429          7.0912 [6.5251, 7.6044]       1.0        [1, 1]         4.0            True
       Howto & Style       590         185          5.4613 [4.8478, 6.0077]       2.0        [2, 5]         7.0            True
              Gaming       101          63          5.1086 [3.8966, 5.8877]       3.0        [2, 7]         1.0           False
      Pets & Animals       136          53          5.0270 [4.2245, 5.5915]       4.0        [2, 7]        13.0           False
           Education       241          57          4.7353 [4.0254, 5.5493]       5.0        [2, 7]         9.0           False
              Comedy       544         141          4.6636 [3.1228, 6.6069]       6.0        [2, 9]         2.0            True
      People & Blogs       490         246          4.3919  [3.601, 5.3158]       7.0        [3, 8]     

**Result with uncertainty.** Music leads and Sports trails, a spread of about 5.6pp around the 3.7331% anchor. But only **one** category holds a fixed position across the bootstrap, and half the rankable categories have rank intervals four or more places wide.

**Read.** The ends of the ranking are firm and the middle is not readable as an ordering. Separately, the reach ranking is a **different** ranking: Spearman 0.367 with a clustered interval of [-0.0154, 0.5474] across 14 categories. That interval covers zero — but its lower bound sits 0.015 from flipping sign, so the honest statement is *unsupported at this number of categories*, not *shown to be absent*. The limit is the count of categories, and no amount of extra video-level data on this dataset fixes it.

**Rule-out.** **Killed: the ranking is really a ranking of a few dominant channels.** Recomputed with one vote per channel — each category's engagement as the median over channels of that channel's own median — the two orderings agree at Spearman **0.9209** (CI [0.8242, 0.9824]), with only two categories moving more than two places. The ordering is a category ordering.

**Answer, hypothesis, confidence.** **Answer:** categories genuinely separate (η² 0.2481), but only the ends of the ranking are firm, and reach and engagement order categories differently. **Hypothesis (associative, and reported live rather than ruled out):** the engagement ordering and the trending-run-length ordering look like the same ordering — Spearman 0.7038 (CI [0.3275, 0.8286], p=0.005). It cannot be the measurement artifact an earlier round found, because engagement here is taken at the first snapshot, before any run length exists. It rests on heavy ties — only 4 distinct integer run-day medians across 14 categories — so it is a live alternative reading of what the ordering orders, not a finding. **Confidence: high** for the levels and the separation verdicts; not for the reach–engagement null.

## Q3 (Compare) — Whether the ranking holds at the reader's own scale

**Question.** Does that category ranking hold among smaller channels, or flip against the larger ones?

**Verb type:** Compare · **Arc role:** BUILD/TEST

**Approach.** Per-category medians within each scale group, the difference with a channel-clustered bootstrap interval, and a rank correlation between the two boards. The multiplicity problem is handled with a **cluster-aware permutation null** — the scale label is permuted across *channels*, with each arm's channel count held fixed — rather than with a video-level test, which would treat 84 videos from one channel as 84 independent observations.

**Assumption check — run before the method.** Cell sizes first: only categories with n≥30 in **both** arms are ranked — 11 qualify and 5 are reported but not ranked. Then independence, which here is entangled with the thing being tested, because the scale group is *itself* defined from a channel-level statistic: ICC 0.743, design effect 4.819, and the worst ranked cell draws 77% of its videos from a single channel.

In [9]:
d = J["q3_scale_holds"]
r = d["rung1_ranking"]
print(f"Spearman(smaller rank, larger rank) = {r['spearman_rho']}  CI {r['spearman_ci95']}  "
      f"P(rho<=0) = {r['spearman_prob_rho_le_0']}")
v = d["rung1_verdict_figure"]
print(f"smaller vs all-ruled anchor: {v['smaller_vs_all_ruled_pp']}pp  CI {v['smaller_vs_all_ruled_ci95']}  <- covers zero")
print(f"smaller vs larger:           {v['smaller_vs_larger_pp']}pp  CI {v['smaller_vs_larger_ci95']}  <- covers zero")
print()
t = pd.DataFrame(r["per_category"])[
    ["category", "n_smaller", "n_larger", "median_smaller", "median_larger", "diff_pp",
     "diff_ci_lo", "diff_ci_hi", "perm_p", "bh_q", "survives_bh_05"]]
print(t.to_string(index=False))
print()
m = d["rung3_multiplicity"]
print(f"raw CI flags {m['categories_with_raw_ci_excluding_zero']} | perm p<.05 "
      f"{m['categories_with_perm_p_below_05']} | expected by chance {m['expected_by_chance']} | "
      f"survive BH {m['categories_surviving_bh_05']} -> {m['surviving_exceptions']}")
print(f"chance alone produces >={m['categories_with_perm_p_below_05']} flags "
      f"{m['p_global_at_least_this_many_by_chance']:.1%} of the time")

Spearman(smaller rank, larger rank) = 0.882  CI [0.464, 0.927]  P(rho<=0) = 0.0
smaller vs all-ruled anchor: -0.198pp  CI [-0.458, 0.125]  <- covers zero
smaller vs larger:           -0.413pp  CI [-1.046, 0.235]  <- covers zero

            category  n_smaller  n_larger  median_smaller  median_larger  diff_pp  diff_ci_lo  diff_ci_hi  perm_p    bh_q  survives_bh_05
               Music        353       432           6.079          8.272   -2.193      -3.319      -1.243 0.00025 0.00275            True
       Howto & Style        348       241           5.202          5.992   -0.790      -1.791       0.415 0.20720 0.56980           False
              Gaming         30        69           5.150          5.016    0.134      -2.276       2.005 0.92677 0.97451           False
           Education        189        50           4.795          4.298    0.497      -2.944       2.362 0.58460 0.71452           False
      People & Blogs        283       197           4.699          4.060    0.639

**Result with uncertainty.** The two boards rank alike at Spearman **0.882** (CI [0.464, 0.927]), and the smaller-channel anchor is statistically indistinguishable from the all-ruled anchor: -0.198pp, CI [-0.458, 0.125]. After multiplicity control, **one** category of eleven differs by scale: Music.

**Read.** The reader's scale is not a different game. Whatever else this study says, it does not say that small channels obey different rules — so the pre-declared fallback fires and this question ships as one line of context plus its exception list, not as a section. That was decided before the numbers were seen, and the freed budget was spent elsewhere.

**Rule-out.** **Partly killed, partly live, and the split is the finding.** At the *list* level the multiple-comparison alternative stands: two of eleven categories cleared p<0.05 against 0.55 expected by chance, and chance alone produces two or more flags 10.1% of the time — so a two-item exception list read off raw intervals is not distinguishable from noise. At the *category* level one exception survives correction anyway (BH q=0.0027). The method that was **rejected** — a video-level Mann–Whitney ignoring clustering — would have produced 4 survivors, and that number is reported precisely to size the over-detection it would have caused.

**Answer, hypothesis, confidence.** **Answer:** the ranking holds at smaller-channel scale; exactly one category differs, and three of the four exception candidates the plan carried do not reproduce — Comedy **reverses sign**. **Hypothesis (associative, to test):** the surviving Music gap is composition inside the category rather than a category effect — 95% of it sits in likes-per-view while comments-per-view is flat across scale board-wide, which is the signature of who is watching rather than what was made. **Confidence: medium** — the verdict is well supported, but the completeness of the exception *list* is not: it rests on 11 points, and there is no evidence on how it behaves at a different scale cut-point, because that sensitivity was the rung-4 check the human refused at the gate.

## Q4 (Explain) — How much of the spread the dials a creator owns actually account for

**Question.** Of the dials owned inside one category — publish slot, tags, comments on/off — which
is most associated with engagement, and **how much spread is left unexplained**?

**Verb type:** Explain · **Arc role:** BREAK

**Approach.** Variance share via the Kruskal–Wallis H statistic, `η²_H = (H − k + 1)/(n − k)`. There
is no `statsmodels` in this environment, so no OLS or GLM table is available — but a rank-based η² is
in any case the right tool for a skewed rate, and the rejected alternative (a linear model's R²)
would have imposed a distributional assumption the data fails.

The **joint** share comes from the cross-classification of the three dials. That cross factor also
absorbs interaction and cell-level noise, so it is an **upper bound** on what the dials jointly
account for — which makes the unexplained share a **lower bound**. That direction matters: it errs
toward the reader rather than toward a flattering headline.

**Assumption check — run before the method, and one of the two forced a method switch.**

1. **Independence.** ICC 0.7566, design effect
   4.54 — videos from one channel are not independent draws; a video-level bootstrap overstates precision by the design effect. Every interval here is a
   channel-clustered bootstrap.
2. **The comments dial has no variance to explain.** The check returns
   **constant — the comments dial has zero variance to explain on engagement rate**; the switch is
   *comments dial measured on like_ratio instead; engagement rate is not defined for comments-disabled videos by construction*.
   This is the single most important check in the run: measured naively, the comments dial would have
   produced a large, entirely artefactual effect.
3. **Spread comparability**, since η²_H is only interpretable if the groups are comparably dispersed.
   The check returns: *engagement-rate spreads are comparable across publish-slot states and only modestly wider for tagged videos, so η²_H is interpretable there; the comments-disabled like-ratio spread is far wider than comments-enabled, which is why the comments line is a median difference with a bootstrap interval and not a mean-based test*

In [10]:
d = J["q4_dial_audit"]
r = d["rung1_smaller_channels"]
print("PER DIAL — the gap and its variance share, always reported together")
for k, v in r["dials"].items():
    verdict = v.get("verdict", v.get("measured_on", ""))
    if "gap_pp" in v:
        print(f"  {v['dial']:44s} gap {v['gap_pp']:+.4f}pp  CI {v['gap_ci95_pp']}  "
              f"eta2 {v['eta2_pct']}%  CI {v['eta2_ci95_pct']}  -> {verdict}")
    else:
        print(f"  {v['dial']:44s} {'(no single gap - 5 bands)':>28s}  "
              f"eta2 {v['eta2_pct']}%  CI {v['eta2_ci95_pct']}")
print()
j = r["joint"]
print(f"JOINT (cross of the three dials, {j['cells_kept_n_ge_20']} cells, n={j['n_used']:,})")
print(f"  eta2_joint = {j['eta2_joint_pct']}%  CI {j['eta2_joint_ci95_pct']}   <- UPPER bound")
h = r["HEADLINE_unexplained_share"]
print(f"  UNEXPLAINED = {h['value_pct']}%  CI {h['ci95_pct']}   <- {h['bound']}")
print()
print("COMPARATORS")
for k, v in r["comparators"].items():
    print(f"  {k:18s} eta2 {v['eta2_pct']:6.3f}%  CI {v['eta2_ci95_pct']}")
print()
print(f"Category alone accounts for {r['comparators']['category']['eta2_pct'] / j['eta2_joint_pct']:.1f}x")
print("as much variance as all three owned dials combined -- and it is the one dimension the reader")
print("has been told they cannot change.")

PER DIAL — the gap and its variance share, always reported together
  publish slot (publish_block, ET)             gap +1.4748pp  CI [0.7925, 1.9561]  eta2 0.728%  CI [0.19, 1.81]  -> live — interval excludes zero
  tags present (has_tags)                      gap +2.1064pp  CI [1.1461, 2.5911]  eta2 1.463%  CI [0.45, 2.9]  -> live — interval excludes zero
  tags (5-band tag_count cut: 0 / 1-8 / 9-16 / 17-28 / 29+)    (no single gap - 5 bands)  eta2 2.428%  CI [1.25, 4.74]
  comments on/off (comments_enabled)           gap -13.2079pp  CI [-20.1896, -7.1158]  eta2 1.456%  CI [0.43, 2.88]  -> like_ratio — NOT engagement rate

JOINT (cross of the three dials, 15 cells, n=3,425)
  eta2_joint = 3.669%  CI [2.56, 6.63]   <- UPPER bound
  UNEXPLAINED = 96.331%  CI [93.37, 97.44]   <- LOWER BOUND

COMPARATORS
  category           eta2 21.020%  CI [17.56, 26.63]
  publish_hour_et    eta2  3.622%  CI [2.46, 7.16]
  publish_dow_et     eta2  0.802%  CI [0.35, 2.07]

Category alone accounts for 5.7

**Result with uncertainty.** The three dials jointly account for at most
**3.669%** (CI [2.56, 6.63]) of the spread in engagement between
videos. The unexplained share is therefore **96.331%** (CI [93.37, 97.44]), and it is a
**lower** bound.

**Read.** The pre-declared conditional did not fire: publish slot is *not* the only dial with an
interval excluding zero — tags clears zero too. But the size of what they explain is the finding.
Category, which this reader cannot change, accounts for
5.7× as much variance as all
three dials they *can* change, combined.

**Rule-out — and this one is not killed. It survives.**

The rival: these are not dials at all, but markers of the channel. A channel that tags carefully,
leaves comments on and publishes on a schedule is plausibly just a better-run channel, and the
"dial" would be a label on the channel rather than a setting on the upload.

The test is a within-channel design. Restrict to channels with videos in **both** states of a dial,
take one paired difference per channel, and see whether the gap survives when the channel is held
fixed. A within-channel contrast also holds category, audience and production standard roughly
constant, because a channel almost always posts in one category.

In [11]:
r3 = J["q4_dial_audit"]["rung3_within_channel"]
p = r3["publish_slot"]
print("PUBLISH SLOT — afternoon minus evening")
print(f"  between videos, all smaller channels : {p['between_video_gap_pp']:+.4f}pp  CI {p['between_video_gap_ci95_pp']}")
print(f"  WITHIN channel, paired ({p['n_channels']} channels): {p['median_paired_diff_pp']:+.4f}pp  CI {p['ci95_pp']}")
print(f"  shrinkage: {p['shrinkage_pct']}%")
print()
print("  Is it just that paired channels are different channels? No --")
print(f"  on those SAME {p['n_channels']} channels the UNPAIRED gap is "
      f"{p['same_channels_unpaired_gap_pp']:+.4f}pp  CI {p['same_channels_unpaired_gap_ci95_pp']}")
print("  i.e. fully present between their videos, and absent once the channel is held fixed.")
print()
print(f"  within-channel UPPER bound {p['ci95_pp'][1]:.4f}pp  <  between-video LOWER bound "
      f"{p['between_video_gap_ci95_pp'][0]:.4f}pp  -> {p['within_channel_ci_excludes_between_video_gap']}")
print()
for k in ["tag_count_median_split", "tags_present"]:
    v = r3[k]
    print(f"{k}: within-channel {v['median_paired_diff_pp']:+.4f}pp  CI {v['ci95_pp']}  "
          f"on {v['n_channels']} channels")

PUBLISH SLOT — afternoon minus evening
  between videos, all smaller channels : +1.4748pp  CI [0.7925, 1.9561]
  WITHIN channel, paired (84 channels): +0.0809pp  CI [-0.1137, 0.5151]
  shrinkage: 94.5%

  Is it just that paired channels are different channels? No --
  on those SAME 84 channels the UNPAIRED gap is +1.1699pp  CI [0.2695, 1.9267]
  i.e. fully present between their videos, and absent once the channel is held fixed.

  within-channel UPPER bound 0.5151pp  <  between-video LOWER bound 0.7925pp  -> True

tag_count_median_split: within-channel +0.0386pp  CI [-0.1593, 0.2863]  on 111 channels
tags_present: within-channel +0.6182pp  CI [-0.5833, 0.83]  on 21 channels


**The rival survives.** The publish-slot gap shrinks **94.5%** when the channel is
held fixed — from +1.4748pp between videos to
+0.0809pp paired, on 84 channels — and the within-channel
interval's upper bound sits *below* the between-video interval's lower bound. It is not a selection
artifact: on those same 84 channels the unpaired gap is
+1.1699pp, fully present.

So the between-video dial gaps are associated with **which channel posted**, not with the setting on
the upload. Note what this does to the headline: a surviving rival here makes the unexplained share
*larger*, not smaller.

**One further line, reported rather than buried.** Comments-disabled videos are excluded from the
engagement rate by construction, so they get their own comparison on like ratio — and the group is
**self-selected**, since disabling comments is a creator's choice.

In [12]:
c = J["q4_dial_audit"]["rung1_smaller_channels"]["dials"]["comments_on_off"]
fc = J["q4_dial_audit"]["fixed_counts"]
print(f"like ratio, comments OFF vs ON (smaller channels)")
for k, v in c.items():
    if k not in ("dial", "note", "selection"):
        print(f"  {k}: {v}")
print()
print(f"  n available: {fc['smaller_comments_disabled_with_like_ratio']} smaller-channel videos with "
      f"comments off AND ratings on")
print(f"  ({fc['smaller_ratings_disabled']} ratings-disabled videos have no like ratio at all)")
print()
print("  SELECTION IS LIVE:", c.get("selection", ""))

like ratio, comments OFF vs ON (smaller channels)
  measured_on: like_ratio — NOT engagement rate
  why: eng_rate is NaN by construction for comments-disabled videos (rate_eligible = not comments_disabled and not ratings_disabled), so comments_enabled is constant True inside the engagement-rate population and has zero variance to explain there. Checked before the method; this is the documented method switch.
  state_medians_pct: {'comments_enabled': 96.9512, 'comments_disabled': 83.7433}
  n_per_state: {'comments_enabled': 3424, 'comments_disabled': 49}
  iqr_per_state_pct: {'comments_enabled': [92.0828, 98.6098], 'comments_disabled': [64.2857, 92.6719]}
  gap_pp: -13.2079
  gap_ci95_pp: [-20.1896, -7.1158]
  eta2_pct: 1.456
  eta2_ci95_pct: [0.43, 2.88]
  kw_H: 51.553
  kw_k: 2
  kw_p_raw: 6.967288801983657e-13

  n available: 49 smaller-channel videos with comments off AND ratings on
  (15 ratings-disabled videos have no like ratio at all)

  SELECTION IS LIVE: LIVE and reported as l

**Answer, hypothesis, confidence.**

**Answer:** the three dials a creator owns jointly account for at most 3.669% of the
spread in engagement between videos; **at least 96.331% is not accounted for by anything
this file shows the creator owning**. Both dials whose intervals exclude zero between videos collapse
to about zero within channel.

**Hypothesis (associative, to test):** the between-video gaps travel with stable channel-level
characteristics — niche, audience composition, production standard — rather than with the per-upload
setting. The confirming test is not available here: it needs either far more paired channels or a
genuine before/after in which one channel deliberately alternates its publish slot across a block of
its own uploads. This data is observational and **nothing here shows that changing a setting changes
engagement.**

**Confidence: medium** for the headline — it is measured two independent ways that agree, it is an
upper-bounded construction so it errs toward the reader, and it reproduces on the all-ruled
comparator. **Low** for the per-dial verdicts, because the named rival was not killed and the
within-channel evidence rests on 84 channels for the strongest dial and fewer for
the others.

## Q5 (Compare) — Which categories carry a priced move, and for how many readers

**Question.** Which packaging and calendar moves inside the existing category are associated with higher engagement, and what does each cost in views?

**Verb type:** Compare · **Arc role:** TURN

**Approach.** Per-category medians in each publish block with channel-clustered bootstrap intervals, and the reach effect expressed as a **signed effect on views** — `+` means MORE views, `−` means FEWER, with the direction written in words as well so nothing depends on a sign being read. Reach is priced twice: at the category's **own** median views first, and at the pooled smaller-channel baseline second, because the pooled baseline understates the most expensive move by about half. This question ships a **coverage statement and no recommendation row** — the decision-owner declined this move at its price once that price was stated correctly.

**Assumption check — run before the method.** The gating check is category mix, and its thresholds were declared **before looking**: material if the chi-square rejects AND (Cramér's V ≥ 0.10 OR total variation distance ≥ 0.10). Observed: V **0.2244**, TVD **0.1414**, permutation p=0.0002 → **the check does not clear**. Cochran's expected-cell rule also failed on the 16×2 table, so the asymptotic chi-square was discarded for a 5,000-shuffle permutation test. Consequence: any figure pooled across categories is a composition artifact before it is anything else, and **the pooled figure may not carry a number**.

In [13]:
d = J["q5_priced_moves"]
rows = []
for cat, v in d["rung1_per_category"].items():
    rows.append({"category": cat, "n_aft": v["n_afternoon"], "n_eve": v["n_evening"],
                 "gap_pp": v["gap_pp"], "ci": v["gap_ci95_pp"], "verdict": v["detection_verdict"],
                 "reach_words": v["reach_pct_in_words"],
                 "reach_own": v["reach_views_per_upload_OWN_BASE_in_words"],
                 "views_per_pt_own": v["views_per_point_own"], "cap": v["cap_label_own_base"].split(" —")[0],
                 "ships_reach": v["SHIPS_reach_figure"]})
print(pd.DataFrame(rows).to_string(index=False))
print()
m = d["method_note"]
print(f"reach COMPUTED for {m['reach_computed_for']} categories, REPORTED for {m['reach_reported_for']}")
print(f"  ({m['reach_ship_rule']})")
print()
c = d["rollup_coverage_statement"]
print("COVERAGE -- these three travel together and LOOKED IN may never appear alone:")
for k in ["looked_in", "found_something", "not_priced"]:
    v = c[k]
    print(f"  {k.upper():16s} {v['videos']:,} of {v['denominator']:,} ({v['pct']}%), {v['categories']} categories")
print(f"  arithmetic: {c['found_something']['arithmetic']}")
print()
print("MULTIPLICITY:", d["method_note"]["comparisons_run"]["multiplicity_outcome"])

       category  n_aft  n_eve  gap_pp                ci        verdict       reach_words                      reach_own  views_per_pt_own      cap  ships_reach
         Comedy     51     50  5.0093 [-1.1246, 7.1844] could-not-tell 48.3% FEWER views 112,212 FEWER views per upload             22401     pass        False
  Entertainment    182     96  0.9866  [0.3155, 1.9076]       detected 20.7% FEWER views  22,491 FEWER views per upload             22797     pass         True
  Howto & Style     80     55 -0.6618  [-1.519, 0.3571] could-not-tell 18.4% FEWER views  26,329 FEWER views per upload             39785 no price        False
          Music     58     34 -0.1186 [-1.8689, 2.7913] could-not-tell 11.7% FEWER views  10,620 FEWER views per upload             89583 no price        False
News & Politics    100     65  0.1335 [-0.3652, 0.5361]        no-tilt  47.8% MORE views   21,475 MORE views per upload            160895 no price        False
 People & Blogs     87     54  2.9046   

**Result with uncertainty.** Two categories of the six carry a detectable gap on the pre-declared interval rule — Entertainment, People & Blogs — not the three the plan's scan had suggested; Comedy drops out once the interval is channel-clustered. Coverage: **LOOKED IN 2,488 of 3,489 (71.3%)**, **FOUND SOMETHING 1,066 (30.6%)**, **NOT PRICED 1,001 (28.7%, 10 categories)**.

**Read.** Both detections rest on the pre-declared uncorrected interval rule. Under Benjamini–Hochberg across the six, **none** of them clears q<0.05. Both numbers are reported and the pre-declared rule was not re-cut to match the corrected one — re-cutting a rule after seeing the result is precisely the failure the pre-declaration exists to prevent. The coverage figure is also knife-edge: drop People & Blogs — detected on a lower bound of +0.075pp — and FOUND SOMETHING falls about 8 points to Entertainment alone (22.24% of smaller-channel videos, from 30.6%). **No seed sweep was run; the swing is the arithmetic of the knife-edge.**

**Rule-out.** **Not killed — it survives, and it is reported live.** The day-of-week rival: publish block and weekday co-vary (Cramér's V 0.1839; weekend publishes are 18.15% of the afternoon block against 25.6% of the evening block). Restricted to Monday–Friday publishes, **neither detected gap remains detectable**, and the magnitude roughly halves in both — so this is not a power story about narrower intervals, it is the effect itself moving.

**Answer, hypothesis, confidence.** **Answer:** the afternoon block is associated with detectably higher engagement in two of six priced categories, reaching 30.6% of smaller-channel videos; both detections fail multiple-comparison correction and neither survives restriction to weekday publishes. **Hypothesis (associative, to test):** the advantage travels with **weekend** publishing rather than with the clock — both gaps roughly double on weekend-only publishes. The confirming test is a four-cell block × weekday-type comparison, which needs about 30 videos per cell and is not answerable on these columns. **Confidence: low**, for three independent reasons: the surviving rival, the failed correction, and reach intervals that straddle zero.

**Two things belong in this notebook and nowhere else.**

First, the middle tier. 3 categories clear
n≥20 but not n≥30. For the reader they collapse into
"not priced", because a direction with no price and no target is a hedge, and hedges are on this
reader's close-the-tab list. Here, where the audience is the BI team, the direction-only figures are
reported.

Second, all six computed reach figures. Only two ship to the reader, attached to the moves that were
actually detected — a reach number printed beside a category that was just told to hold its calendar
is the opposite instruction, printed by us. Nothing is hidden: the other four are below.

In [14]:
d = J["q5_priced_moves"]
mt = d["middle_tier_n20_29_NOTEBOOK_ONLY"]
print(f"MIDDLE TIER, n 20-29 -- NOTEBOOK ONLY ({mt['tag']})")
print(pd.DataFrame(mt["categories"]).T.to_string())
print()
print("ALL SIX REACH FIGURES -- computed for six, reported for two")
rows = []
for cat, v in d["rung1_per_category"].items():
    rows.append({"category": cat, "verdict": v["detection_verdict"],
                 "reach_%": v["reach_pct_signed"], "in_words": v["reach_pct_in_words"],
                 "own_base_views": v["reach_views_per_upload_OWN_BASE_signed"],
                 "pooled_base_views": v["reach_views_per_upload_POOLED_BASE_signed"],
                 "SHIPS": v["SHIPS_reach_figure"]})
print(pd.DataFrame(rows).to_string(index=False))
print()
b = d["rollup_pooled_figure_BARRED"]
print(f"POOLED FIGURE: barred from the verdict layer; may_ship_with_number = {b['may_ship_with_number']}")
print(f"  {b['strata_running_against_the_aggregate']['count']} of "
      f"{b['strata_running_against_the_aggregate']['of']} strata run against the aggregate "
      f"({', '.join(b['strata_running_against_the_aggregate']['categories'])})")
print(f"  and {d['rollup_block_other']['pct_of_smaller']}% of smaller-channel videos publish in NEITHER block.")

MIDDLE TIER, n 20-29 -- NOTEBOOK ONLY (R3 C1 — collapsed into NOT PRICED for the reader; may appear in analysis.ipynb only. Direction with no price and no target is a hedge.)
                     n_afternoon n_evening  gap_pp            ci95_pp videos
Film & Animation              29        24 -0.3771  [-4.1523, 2.4057]    131
Science & Technology          48        28  0.3285   [-1.8188, 2.217]    224
Sports                        28        58  0.4099  [-0.2156, 1.1243]    210

ALL SIX REACH FIGURES -- computed for six, reported for two
       category        verdict  reach_%          in_words  own_base_views  pooled_base_views  SHIPS
         Comedy could-not-tell   -48.34 48.3% FEWER views         -112212             -57538  False
  Entertainment       detected   -20.66 20.7% FEWER views          -22491             -24587   True
  Howto & Style could-not-tell   -18.43 18.4% FEWER views          -26329             -21937  False
          Music could-not-tell   -11.66 11.7% FEWER view

## Q6 (Recommend) — What separated the channels that trended twice, on their first video

**Question.** Among smaller channels, what separates those that trended more than once from one-hit channels — and what of that can a creator actually copy?

**Verb type:** Recommend · **Arc role:** RESOLVE

**Approach.** The obvious comparison is circular: compare all of a repeat channel's videos against one-hit videos and the outcome is measured *after* the repeating it is supposed to explain. So the design compares each repeat channel's **first** trending video — measured before any repeating had happened — against the one-hit channels' only video. The circular version is computed too, but only to show what the design removes; it is barred from every register.

**Assumption check — run before the method.** Three checks. **Design:** 450 of 450 flagged videos are strictly the channel's earliest trending day, with 0 ties — so the tie-break never fired and the comparison is genuinely pre-repeat. **Independence:** one video per channel on both sides, so the video bootstrap IS a channel bootstrap. The channel-clustering problem that binds Q3/Q4/Q5 does not bind this comparison — a genuine strength of the design. **Shape:** normality REJECTED on both sides (D'Agostino p=4.7e-18 group A, 2.1e-51 group B), both right-skewed (+1.10, +1.44), means 1.11x and 1.40x their medians. Equal variance was NOT rejected (Brown-Forsythe p=0.695) — that check passed and is not what forces the switch; the distribution SHAPE is, and the mean/median divergence differs between the two groups so a mean comparison would measure tail asymmetry as well as location. Chosen: median difference with a 10,000-resample percentile bootstrap CI. Rejected: difference of means with a t-interval.

In [15]:
d = J["q6_repeat_vs_onehit"]
h, c, g = d["headline_gap"], d["conversion"], d["group_sizes"]
print(f"repeat channels' FIRST video  {h['group_A_median_pct']:.4f}%   n={h['group_A_n']}")
print(f"one-hit channels' only video  {h['group_B_median_pct']:.4f}%   n={h['group_B_n']}")
print(f"GAP  {h['gap_pp']:+.4f}pp   CI {[round(x, 4) for x in h['gap_ci95_pp']]}   "
      f"Cliff's delta {h['cliffs_delta']:.3f} {[round(x, 3) for x in h['cliffs_delta_ci95']]}")
print(f"  n note: {d['headline_gap']['n_correction']['correct']} -- "
      f"the plan wrote {d['headline_gap']['n_correction']['PRD_wrote']}; "
      f"{d['headline_gap']['n_correction']['reason']}")
print()
print(f"COUNTABLE STAKES  {c['extra_reactions_per_upload']:,.0f} extra likes+comments per upload "
      f"({c['extra_reactions_per_upload_ci95'][0]:,.0f}-{c['extra_reactions_per_upload_ci95'][1]:,.0f})")
print(f"GROUPS  {g['repeat_channels']} repeat ({g['repeat_pct_of_smaller_channels']}%) vs "
      f"{g['one_hit_channels']} one-hit ({g['one_hit_pct_of_smaller_channels']}%) of {g['channels_denominator']:,}")
print()
b = d["barred_pooled_comparison"]
print(f"THE CIRCULAR VERSION, computed to show what the design removes -- BARRED (barred={b['barred']}):")
print(f"  all repeat-channel videos {b['pooled_repeat_videos_median_pct']:.4f}% vs one-hit "
      f"{b['one_hit_median_pct']:.4f}% = {b['pooled_gap_pp']:+.4f}pp")
print(f"  the first-video design gives {h['gap_pp']:+.4f}pp -- the circular framing UNDERSTATES it")

repeat channels' FIRST video  4.8391%   n=445
one-hit channels' only video  3.0451%   n=899
GAP  +1.7940pp   CI [1.3164, 2.2044]   Cliff's delta 0.225 [0.164, 0.288]
  n note: n=445 vs 899 -- the plan wrote n=445 vs 935; 935 is the one-hit CHANNEL count; 36 of those videos have comments or ratings disabled and carry no engagement rate

COUNTABLE STAKES  2,135 extra likes+comments per upload (1,567-2,624)
GROUPS  450 repeat (32.5%) vs 935 one-hit (67.5%) of 1,385

THE CIRCULAR VERSION, computed to show what the design removes -- BARRED (barred=True):
  all repeat-channel videos 3.7238% vs one-hit 3.0451% = +0.6787pp
  the first-video design gives +1.7940pp -- the circular framing UNDERSTATES it


**Result with uncertainty.** Repeat channels' first trending video scored **4.84%** against **3.05%** for one-hit channels — a gap of **+1.7940pp** (95% CI [1.3164, 2.2044]), on 445 versus 899 videos. At the smaller-channel baseline that is about **2,135 extra likes and comments per upload**.

**Read.** The separation existed before any repeating had happened. That does not establish which way it runs, and the notebook does not pretend otherwise — see the rule-out.

**Rule-out.** **Killed: the gap is a category-mix artifact.** Repeat channels might simply concentrate in categories that engage more. Standardising both groups to a common category mix retains **74.4%** of the gap — 1.336pp, CI [0.76, 1.962], still excluding zero. Six of six within-category point estimates are positive, though only one excludes zero individually: the standardised result is carried by consistency of sign, not by any single category.

**Reported live, not assumed away:** the reverse reading. The first-video design rules out 'repeating created the gap' — group A's video was measured before any repeating had happened. It does NOT rule out the reverse: channels that were already going to do well may engage more from the start AND trend again. The direction of association between channel strength and engagement is not resolvable in this data and no experiment is available. This is a limit and it ships.

### Rung 4 — the descriptors, and a fallback declared before any of this ran

The gap is real and it is measured before any repeating. The open question is what, if anything, a
creator could copy from it. Tag count was the only concrete descriptor the plan carried, and it was
demoted out of the recommendation set early on — "use tags" is not a next action for this reader.

So five further descriptors were pre-declared, along with a **binding fallback**: if none of them
separates the two groups, the report may not fall back on tags, and the verdict becomes a bounded
negative. That was written before the numbers existed, precisely so it could not be fitted afterwards.

One of the five cannot be scored the same way. **Cadence** — days between a channel's trending videos
— is *structurally undefined* for a one-hit channel, which has no second video. That is an
impossibility, not evidence of no difference, so it is described rather than tested, and excluded
from the fallback trigger.

In [16]:
d = J["q6_repeat_vs_onehit"]
rows = []
for name, v in d["descriptors"].items():
    if name == "cadence":
        rows.append({"descriptor": name, "separates": str(v["separates"]), "levels": "-",
                     "best_level": "-", "diff_pp": "-", "ci": "-", "p_BH": "-"})
        continue
    L = v["largest_absolute_level"]
    rows.append({"descriptor": name, "separates": str(v["separates"]),
                 "separates_CI_rule": str(v["separates_ci_only_uncorrected"]),
                 "levels": v["n_levels_tested"], "best_level": L["level"],
                 "diff_pp": round(L["diff_pp"], 3), "ci": [round(x, 2) for x in L["ci"]],
                 "p_BH": round(L["p_bh"], 4)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print(f"evaluable descriptors      {d['descriptors_evaluable_count']}  (cadence excluded: "
      f"{d['descriptors']['cadence']['separates']})")
print(f"separate on pre-declared CI rule : {d['descriptors_separating_count_ci_only_uncorrected']} "
      f"-> {d['descriptors_that_separate_ci_only_uncorrected']}")
print(f"also clear Benjamini-Hochberg    : {d['descriptors_separating_count']} "
      f"-> {d['descriptors_that_separate']}")
print(f"FALLBACK FIRED = {d['fallback_fired']}   ({d['fallback_rule']})")
print()
print("cadence, described rather than tested:", d["descriptors"]["cadence"]["repeat_group_description"])
print()
print("SENSITIVITY -- does the headline gap hold up?")
s = d["sensitivity"]
sv = s if isinstance(s, dict) else {}
for k, v in list(sv.items())[:20]:
    if isinstance(v, dict) and "gap_pp" in v:
        print(f"  {k:44s} {v['gap_pp']:+.4f}pp  CI {[round(x, 2) for x in v.get('ci95_pp', [float('nan')] * 2)]}")
ss = d["sensitivity_summary"]
print(f"  -> {ss['cuts_where_point_estimate_positive']}/{ss['cuts_run']} cuts positive; "
      f"{ss['cuts_where_gap_ci_excludes_zero']}/{ss['cuts_run']} also exclude zero")

           descriptor              separates separates_CI_rule levels    best_level diff_pp            ci    p_BH
     publish_slot_mix                  False              True     10         other   5.777 [0.25, 11.16]  0.0876
         category_mix                   True              True     16 Howto & Style    6.48  [3.16, 10.2]  0.0009
comments_enabled_rate                  False              True      1          True   2.097  [0.35, 3.58]  0.0617
 description_presence                   True              True      1          True   4.796  [3.09, 6.45]  0.0003
              cadence structurally undefined               NaN      -             -       -             -       -

evaluable descriptors      4  (cadence excluded: structurally undefined)
separate on pre-declared CI rule : 4 -> ['publish_slot_mix', 'category_mix', 'comments_enabled_rate', 'description_presence']
also clear Benjamini-Hochberg    : 2 -> ['category_mix', 'description_presence']
FALLBACK FIRED = False   (pre-decla

**Answer, hypothesis, confidence.**

**Answer:** channels that went on to trend more than once were **already separated on their very
first trending video**, before any repeating had happened — 4.84% against
3.05%, a gap of +1.7940pp (CI
[1.3164, 2.2044]), worth about
2,135 reactions on a median upload. The gap survives category
standardisation and all 16 sensitivity cuts.

The pre-declared fallback **did not fire**: of the four descriptors that can be tested, all four
separate on the pre-declared interval rule and two also clear multiple-comparison correction. But
the two that clear correction are a **composition** difference (category mix) and a **near-ceiling**
difference (description presence, 99.56% against 94.76% — two videos versus forty-nine), and the two
closest to an actual per-upload choice do not clear correction at all. What that means for the
report's verdict is a question about the plan's own rules, not about these numbers, and it has been
escalated rather than settled here.

**Hypothesis (associative, to test):** the first-video gap is associated with an audience that
existed **before** the trending appearance rather than with anything in the packaging — a channel
arriving with a formed audience shows more reaction per view on arrival and also has the standing to
arrive again. The confirming test names the missing column exactly: condition the gap on channel
subscriber count or any pre-trending baseline. Neither exists in this dataset, so the test needs the
platform's channel endpoint. **This is the same mechanism as the live reverse association, and it
cannot be settled here.**

**Confidence: high** for the headline gap — the design is verified strictly non-circular,
independence holds at the unit of analysis, the category-mix rival is dead, and every sensitivity cut
is positive. **Medium** for the descriptor layer. **Low** for the two descriptors that detect on the
interval rule but miss correction.

---

## Conclusion

**One-hit or repeat — what separates a small channel that trends twice?**

On this board, over these 205 days: the channels that came back were already different when they
arrived. Their **first** trending video — the one that existed before there was any pattern to
detect — scored 4.84% engagement against 3.05%
for the channels that trended once and were not seen again. That is +1.7940pp
(CI [1.3164, 2.2044]), about
2,135 extra likes and comments on a median upload, and it holds
after standardising both groups to a common category mix and across all
16 sensitivity cuts that were run.

**What carries that answer, and what does not.** Q6 carries it, on a design built specifically to
avoid measuring the consequence of repeating instead of its antecedent. Q3 licenses it: the reader's
scale is not a different game, so a finding measured on smaller channels speaks to smaller channels.
Q1 sizes it against a baseline whose spread is wide enough that a two-point gap is a real but modest
shift inside a very broad distribution.

**What the analysis cannot conclude, stated as plainly as what it can.** It cannot tell you which
direction the association runs. The first-video design rules out the possibility that repeating
created the gap — but not the possibility that channels already positioned to do well engage more
from the start *and* come back. Settling that needs a column this dataset does not have, and no
amount of care with the columns it does have substitutes.

Nor can it hand the reader a mechanism. Q4 is where the study breaks, and it breaks honestly: the
three dials a creator owns account for at most 3.669% of the spread between videos,
so **at least 96.331% is not accounted for by anything visible here that they control** —
and both dials with a real between-video gap collapse to approximately zero once the channel is held
fixed. Category, which they have been told they cannot change, accounts for
5.7× as much.

**Three questions, three populations, three methods, one shape.** Q1's intra-channel correlation of
0.7779; Q4's 94.5% shrinkage
when the channel is held fixed; Q6's gap present on the very first video. Independently, all three
locate the variation **between channels rather than within them**. That is an arithmetic statement
about where variance sits, not a claim about what causes it — and it is the most defensible thing
this study found.

The reader arrived believing that trending is mostly outside their control. The honest verdict is
that on the evidence here they are more right than the framing of the question assumed — but the
part that is not luck looks like a property of the channel rather than a setting on the upload, and
this data cannot say how a channel comes to have it.

*What follows from this for the reader's next upload is the storyteller's layer, and it is not
written here.*

## Limits, and what I would do next

**Where this is fragile.**

- **The direction of the central association is unresolved**, and it is the single largest limit.
  Every framing available on these columns is compatible with both readings.
- **No control group exists at any framing.** Every row already reached the board; the uploads that
  sank were never collected. So every statement is scoped to *videos that trend*, and nothing here
  predicts whether a video trends.
- **Q5's findings are the weakest in the file** and are labelled so: neither detected gap survives
  restriction to weekday publishes, none of the six clears multiple-comparison correction, and the
  coverage share rests on a knife edge — drop People & Blogs, detected on a
  +0.075pp lower bound, and FOUND SOMETHING
  falls about 8 points to
  Entertainment alone. No seed sweep was run; the swing is the arithmetic of the knife-edge.
- **The exception list in Q3 is not complete.** Chance alone produces two or more flags
  10.1% of the time, and there is no evidence on how the
  list behaves at a different scale threshold, because that check was refused at the gate and
  reallocated to Q6.
- **Selection is not evenly spread.** The rate exclusion is roughly ten times heavier in News &
  Politics than in Music, and it lands on a category that also sits near the bottom of the ranking —
  so that row is measured only on the videos whose creators left reactions switched on.
- **Both window edges are partial**, so trending-run length is censored for
  4.61%
  of smaller-channel videos and is reported as context, never as an outcome.
- **One region, seven months, one platform.** No seasonal replication is possible.

**What I would do next, in order.**

1. **Join channel subscriber count** from the platform's channel endpoint and condition the Q6 gap on
   it. This is the one test that would move the central question, and it is a join rather than a new
   study.
2. **Run the within-channel design at scale.** Q4's strongest result rests on 84
   channels. More trending days, or more regions, would either confirm that the dials are channel
   markers or find the subset where they are not.
3. **Test the run-length reading of Q2** by associating per-video first-snapshot engagement with that
   video's eventual run days *within* category. If it survives within category it is a durability
   association; if it vanishes it is an artifact of fourteen aggregate points.
4. **Settle Q5's weekend hypothesis** with a four-cell block × weekday-type comparison. It needs about
   30 videos per cell, which this window cannot supply and a longer one could.

## Appendix — findings synthesis and the integrity gate

`findings.md` is generated by `scripts/17_build_findings.py` from `out/*.json`; the generator is the
root and a hand-edit to the markdown dies on the next rebuild. The cell below does not rebuild it —
it **verifies** that the notebook's prose still matches the JSON it was interpolated from.

This is the guard against the quietest failure mode available to a generated notebook: someone
re-runs a question script, a number moves, and the notebook keeps displaying the figure that was
true at build time. The assertions below fail loudly instead.

In [17]:
# Values frozen into this notebook's prose AT BUILD TIME, re-checked against out/*.json now.
EXPECTED = {
    "q6_gap_pp": 1.794032021444317,
    "q6_ci_lo": 1.3163755889100224,
    "q6_ci_hi": 2.2044215066147315,
    "q6_group_a_n": 445,
    "q6_group_b_n": 899,
    "q4_unexplained_pct": 96.331,
    "q4_joint_eta2_pct": 3.669,
    "q4_shrinkage_pct": 94.5,
    "q5_found_something_videos": 1066,
    "q5_looked_in_videos": 2488,
    "q1_smaller_median": 3.5349,
    "smaller_videos": 3489,
    "smaller_channels": 1385,
    "channels_denominator": 2099,
    "days_to_trend_max_pre_exclusion": 4215,
    "days_to_trend_max_ruled": 28,
}

ACTUAL = {
    "q6_gap_pp": J["q6_repeat_vs_onehit"]["headline_gap"]["gap_pp"],
    "q6_ci_lo": J["q6_repeat_vs_onehit"]["headline_gap"]["gap_ci95_pp"][0],
    "q6_ci_hi": J["q6_repeat_vs_onehit"]["headline_gap"]["gap_ci95_pp"][1],
    "q6_group_a_n": J["q6_repeat_vs_onehit"]["headline_gap"]["group_A_n"],
    "q6_group_b_n": J["q6_repeat_vs_onehit"]["headline_gap"]["group_B_n"],
    "q4_unexplained_pct": J["q4_dial_audit"]["rung1_smaller_channels"]["HEADLINE_unexplained_share"]["value_pct"],
    "q4_joint_eta2_pct": J["q4_dial_audit"]["rung1_smaller_channels"]["joint"]["eta2_joint_pct"],
    "q4_shrinkage_pct": J["q4_dial_audit"]["rung3_within_channel"]["publish_slot"]["shrinkage_pct"],
    "q5_found_something_videos": J["q5_priced_moves"]["rollup_coverage_statement"]["found_something"]["videos"],
    "q5_looked_in_videos": J["q5_priced_moves"]["rollup_coverage_statement"]["looked_in"]["videos"],
    "q1_smaller_median": J["q1_baseline"]["rung1_estimates"]["smaller"]["eng_rate_pct"]["median"],
    "smaller_videos": J["basis"]["smaller_channels"]["videos"],
    "smaller_channels": J["basis"]["smaller_channels"]["channels"],
    "channels_denominator": J["basis"]["population"]["channels_ruled_POST_exclusion"],
    "days_to_trend_max_pre_exclusion": J["basis"]["population"]["days_to_trend_max_pre_exclusion"],
    "days_to_trend_max_ruled": J["basis"]["all_ruled"]["days_to_trend_max"],
}

drift = {k: (v, ACTUAL[k]) for k, v in EXPECTED.items() if abs(float(v) - float(ACTUAL[k])) > 1e-9}
if drift:
    for k, (want, got) in drift.items():
        print(f"DRIFT  {k}: notebook says {want}, out/*.json says {got}")
    raise AssertionError(
        f"{len(drift)} figure(s) drifted. A question script was re-run without rebuilding the "
        "notebook. Run scripts/18_build_notebook.py, then re-execute."
    )
print(f"INTEGRITY OK - all {len(EXPECTED)} interpolated figures still match out/*.json")

INTEGRITY OK - all 16 interpolated figures still match out/*.json


## Appendix — reproducibility

In [18]:
import sys, platform
print("python  ", sys.version.split()[0], "on", platform.platform())
for m in (np, pd):
    print(f"{m.__name__:8s}", m.__version__)
import scipy
print("scipy   ", scipy.__version__)
print()
print("SEEDS — every bootstrap and permutation in this study is seeded; re-running reproduces exactly.")
print("DATA AS OF — US trending board 2017-11-14 to 2018-06-14, 205 of 213 calendar days present.")
print()
print("RE-RUN ORDER")
for s in ["08_clean.py", "09_shape.py", "10_basis_check.py", "11_q1_baseline.py",
          "12_q2_category_rank.py", "13_q3_scale_holds.py", "14_q4_dial_audit.py",
          "15_q5_priced_moves.py", "16_q6_repeat_vs_onehit.py", "17_build_findings.py",
          "18_build_notebook.py", "19_execute_notebook.py"]:
    print("  python3 2-analysis/scripts/" + s)
print()
print("Scripts 11-16 are independent of each other and may run in parallel.")
print("Every script resolves its paths from Path(__file__).resolve().parents[2] — no cwd assumption.")

python   3.14.4 on macOS-26.5.2-arm64-arm-64bit-Mach-O
numpy    2.5.1
pandas   3.0.3
scipy    1.18.0

SEEDS — every bootstrap and permutation in this study is seeded; re-running reproduces exactly.
DATA AS OF — US trending board 2017-11-14 to 2018-06-14, 205 of 213 calendar days present.

RE-RUN ORDER
  python3 2-analysis/scripts/08_clean.py
  python3 2-analysis/scripts/09_shape.py
  python3 2-analysis/scripts/10_basis_check.py
  python3 2-analysis/scripts/11_q1_baseline.py
  python3 2-analysis/scripts/12_q2_category_rank.py
  python3 2-analysis/scripts/13_q3_scale_holds.py
  python3 2-analysis/scripts/14_q4_dial_audit.py
  python3 2-analysis/scripts/15_q5_priced_moves.py
  python3 2-analysis/scripts/16_q6_repeat_vs_onehit.py
  python3 2-analysis/scripts/17_build_findings.py
  python3 2-analysis/scripts/18_build_notebook.py
  python3 2-analysis/scripts/19_execute_notebook.py

Scripts 11-16 are independent of each other and may run in parallel.
Every script resolves its paths from Path(